In [1]:
# ============================================================
# FILE — FINAL RAW OPENLANDMAP OC EXPORT
# ------------------------------------------------------------
# PAN-INDIA SOIL HEALTH MAPPING
#
# PURPOSE
# ─────────────────────────────────────────────────────────────
# Export a Pan-India OpenLandMap Organic Carbon baseline layer
# for comparison with the RF-based OC prediction map.
#
# FINAL FIXES INCLUDED
# ─────────────────────────────────────────────────────────────
# ✅ NO LULC masking
#    -> Produces a raw OpenLandMap OC baseline layer.
#
# ✅ Correct OC unit conversion handled here directly
#    -> Previous pipeline exported raw / 10 and app multiplied
#       by 5.0, which is equivalent to raw / 2.
#    -> This script exports raw / 2 directly.
#
# ✅ Output band renamed to "OC_OLM"
#
# ✅ Explicit CRS + 30 m global transform
#    -> Consistent with the validated RF export grid.
#
# ✅ Skip export if asset already exists
#
# ✅ Robust Earth Engine initialization and clean summary
#
# IMPORTANT
# ─────────────────────────────────────────────────────────────
# • This is a comparison baseline, not an RF prediction.
# • No LULC mask is applied.
# • Masked/no-data pixels remain masked.
# • The final GEE app should load this asset directly with
#   NO extra multiply(5.0).
# ============================================================

import ee
import yaml


# ============================================================
# 1. LOAD CONFIGURATION
# ============================================================

print("Loading configuration...")

with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

PROJECT_ID = config["gee"]["project_id"]


# ============================================================
# 2. EXPORT SETTINGS
# ============================================================

ASSET_NAME = "PanIndia_OLM_OC_Raw"
ASSET_ID = f"projects/{PROJECT_ID}/assets/{ASSET_NAME}"

EXPORT_CRS = "EPSG:4326"

# Same global ~30 m grid definition used for the corrected
# RF raw prediction exports.
EPSG4326_30M_TRANSFORM = [
    0.000269494585235856472,
    0,
    -180.0,
    0,
    -0.000269494585235856472,
    90.0
]

MAX_PIXELS = 1e13


# ============================================================
# 3. INITIALIZE EARTH ENGINE
# ============================================================

print("\nInitializing Earth Engine...")

try:
    ee.Initialize(
        project=PROJECT_ID,
        opt_url="https://earthengine-highvolume.googleapis.com"
    )
    print("✅ Earth Engine initialized successfully")

except Exception as e:
    print("Earth Engine initialization failed. Starting authentication...")
    print(e)

    ee.Authenticate()

    ee.Initialize(
        project=PROJECT_ID,
        opt_url="https://earthengine-highvolume.googleapis.com"
    )

    print("✅ Earth Engine authenticated and initialized")


# ============================================================
# 4. DEFINE PAN-INDIA REGION OF INTEREST
# ============================================================

print("\n⏳ Loading India boundary...")

india_boundary = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "India"))
)

roi = india_boundary.geometry()

print("✅ India boundary loaded")


# ============================================================
# 5. LOAD AND PREPARE RAW OPENLANDMAP OC
# ============================================================

print("\n⏳ Preparing raw OpenLandMap Organic Carbon layer...")

# OpenLandMap Organic Carbon surface layer:
#   Dataset band : b0
#
# Conversion used here:
#   Previous pipeline exported raw / 10,
#   then app multiplied by 5.0.
#
#   raw / 10 × 5 = raw / 2
#
# Therefore:
#   Correct final export = raw / 2
#
# The app should later read this exported raster directly,
# without any additional multiply(5.0).

olm_oc_raw = (
    ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02")
    .select("b0")
    .divide(2.0)
    .rename("OC_OLM")
    .toFloat()
    .clip(roi)
    .set({
        "layer_name": "OpenLandMap Organic Carbon Raw Baseline",
        "units": "percent_OC",
        "conversion_note": "Exported as native encoded OLM OC divided by 2.0; replaces previous divide(10) then app multiply(5) workflow.",
        "masking": "No LULC masking applied",
        "source_dataset": "OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02"
    })
)

print("✅ Raw OpenLandMap OC layer prepared")
print("   Mask       : None")
print("   Output band : OC_OLM")
print("   Conversion  : raw / 2.0")


# ============================================================
# 6. CHECK WHETHER OUTPUT ASSET ALREADY EXISTS
# ============================================================

print("\n⏳ Checking if asset already exists...")
print(f"   Asset ID: {ASSET_ID}")

asset_exists = False

try:
    ee.data.getAsset(ASSET_ID)
    asset_exists = True

except ee.EEException:
    asset_exists = False


# ============================================================
# 7. EXPORT IF ASSET DOES NOT EXIST
# ============================================================

if asset_exists:

    print("\n⏭️ Asset already exists — skipping export.")
    print("   To re-export, either delete the existing asset or change ASSET_NAME.")
    print(f"   Existing asset: {ASSET_ID}")

else:

    print("✅ Asset does not exist — proceeding with export.")

    print(f"\n🚀 Submitting export task → {ASSET_ID}")

    try:
        task = ee.batch.Export.image.toAsset(
            image=olm_oc_raw,
            description=f"Export_{ASSET_NAME}",
            assetId=ASSET_ID,
            region=roi,
            crs=EXPORT_CRS,
            crsTransform=EPSG4326_30M_TRANSFORM,
            maxPixels=MAX_PIXELS
        )

        task.start()

        print("✅ Task submitted successfully!")
        print()
        print(f"   Asset Name : {ASSET_NAME}")
        print(f"   Asset ID   : {ASSET_ID}")
        print("   Region     : India boundary")
        print("   CRS        : EPSG:4326")
        print("   Grid       : common global ~30 m transform")
        print("   Band       : OC_OLM")
        print("   Units      : % OC")
        print("   Conversion : raw / 2.0")
        print("   Mask       : None")
        print()
        print("   Monitor task progress at:")
        print("   https://code.earthengine.google.com/tasks")

    except Exception as e:
        print("❌ Task submission failed:")
        print(e)
        raise

/opt/homebrew/anaconda3/envs/shc-env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Loading configuration...

Initializing Earth Engine...
✅ Earth Engine initialized successfully

⏳ Loading India boundary...
✅ India boundary loaded

⏳ Preparing raw OpenLandMap Organic Carbon layer...
✅ Raw OpenLandMap OC layer prepared
   Mask       : None
   Output band : OC_OLM
   Conversion  : raw / 2.0

⏳ Checking if asset already exists...
   Asset ID: projects/ee-mtpictd-ratinder-mcs/assets/PanIndia_OLM_OC_Raw
✅ Asset does not exist — proceeding with export.

🚀 Submitting export task → projects/ee-mtpictd-ratinder-mcs/assets/PanIndia_OLM_OC_Raw
✅ Task submitted successfully!

   Asset Name : PanIndia_OLM_OC_Raw
   Asset ID   : projects/ee-mtpictd-ratinder-mcs/assets/PanIndia_OLM_OC_Raw
   Region     : India boundary
   CRS        : EPSG:4326
   Grid       : common global ~30 m transform
   Band       : OC_OLM
   Units      : % OC
   Conversion : raw / 2.0
   Mask       : None

   Monitor task progress at:
   https://code.earthengine.google.com/tasks
